In [ ]:
import pandas as pd
import json

# Load the raw JSON dataset that contains mandi price and MSP records.
# This is the starting point before any cleaning or standardization.
with open("track3_price_and_msp.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))

<class 'list'>


In [ ]:
# Inspect the raw JSON structure to confirm it is a list of records
# and to see the first rows before converting it into a DataFrame.
print(data if isinstance(data, dict) else data[:2])

[{'record_id': 'PR001650', 'date': '2026/07/26', 'mandi_id': 'MANDI005', 'district': None, 'crop_name': 'Kapas', 'min_price': '6850.985817820893', 'max_price': '₹7,570.17', 'modal_price': 7210.58, 'msp': '₹6,620.00'}, {'record_id': 'PR000246', 'date': '2026-07-17', 'mandi_id': 'MANDI028', 'district': 'Fatehabad', 'crop_name': 'कपास', 'min_price': '₹6,944.79', 'max_price': 7654.18, 'modal_price': 'Rs. 7,299', 'msp': ''}]


In [ ]:
# Convert the raw list of JSON records into a DataFrame for easier cleaning.
# This gives us a tabular structure to inspect and transform systematically.
df = pd.DataFrame(data)

print("Shape:", df.shape)
display(df.head())

Shape: (12000, 9)


,record_id,date,mandi_id,district,crop_name,min_price,max_price,modal_price,msp
0,PR001650,2026/07/26,MANDI005,NaN,Kapas,6850.985817820893,"₹7,570.17",7210.58,"₹6,620.00"
1,PR000246,2026-07-17,MANDI028,Fatehabad,कपास,"₹6,944.79",7654.18,"Rs. 7,299",
2,PR010091,09.01.2026,MANDI011,Jalandhar,Dhaan,"Rs. 1,857",2013.18,"₹1,935.06","INR 2,183"
3,PR000982,2026/05/24,M012,Ferozepur,corn,"₹1,873.34","Rs. 1,986","Rs. 1,930","Rs. 2,090"
4,PR001708,18/08/2026,013,Karnal,Cotton,"5,878.62/-",6574.283051769212,6226.45,"Rs. 6,620"


In [ ]:
# Check the dataset schema and identify missing values before any cleaning.
# This step helps determine which columns need standardization or imputation.
print("Column information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Column information:
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   record_id    12000 non-null  str   
 1   date         12000 non-null  str   
 2   mandi_id     10765 non-null  str   
 3   district     11227 non-null  str   
 4   crop_name    12000 non-null  str   
 5   min_price    12000 non-null  object
 6   max_price    12000 non-null  object
 7   modal_price  12000 non-null  object
 8   msp          12000 non-null  object
dtypes: object(4), str(5)
memory usage: 843.9+ KB

Missing values:
record_id         0
date              0
mandi_id       1235
district        773
crop_name         0
min_price         0
max_price         0
modal_price       0
msp               0
dtype: int64

Duplicate rows: 0


In [ ]:
# Inspect the raw price fields to see how currency values are formatted.
# This is important because prices may contain symbols, units, or missing entries.
price_columns = ["min_price", "max_price", "modal_price", "msp"]

for col in price_columns:
    print(f"\n--- {col} ---")
    print(df[col].head(15).to_list())


--- min_price ---
['6850.985817820893', '₹6,944.79', 'Rs. 1,857', '₹1,873.34', '5,878.62/-', '', '₹2,270.14', '₹1,961.63', 2277.53, '₹3,156.94', 'Rs. 1,866', '', '1989.0202736774606', 'INR 2,016', '₹3,391.89']

--- max_price ---
['₹7,570.17', 7654.18, 2013.18, 'Rs. 1,986', '6574.283051769212', '₹7,324.58', '2823.4429693584902', 'Rs. 2,340', 'INR 2,458', 3890.02, 'INR 2,193', 'Rs. 6,850', '2,138.04/-', 'Rs. 2,300', '₹4,215.91']

--- modal_price ---
[7210.58, 'Rs. 7,299', '₹1,935.06', 'Rs. 1,930', 6226.45, '', '2,546.79/-', 'INR 2,151', '₹2,367.64', '', 'Rs. 2,029', 'Rs. 6,290', '₹2,063.53', 2158.4, 'INR 3,804']

--- msp ---
['₹6,620.00', '', 'INR 2,183', 'Rs. 2,090', 'Rs. 6,620', '6,620.00/-', '₹2,275.00', 2275, '2,275.00/-', 'Rs. 3,500', 'Rs. 2,090', 'Rs. 5,650', '2183', 'Rs. 2,275', 'Rs. 3,500']


In [ ]:
# Clean all price fields by removing currency symbols and converting strings to numeric values.
# This standardizes the price columns so they can be used for analysis and comparison.
price_columns = ["min_price", "max_price", "modal_price", "msp"]

for col in price_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace("INR", "", regex=False)
        .str.replace("/-", "", regex=False)
        .str.strip()
    )
    
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# Validate the cleaned price columns to confirm they are numeric and complete enough for analysis.
print("Price columns after cleaning:\n")

for col in price_columns:
    print(f"{col}:")
    print("Data type:", df[col].dtype)
    print("Missing values:", df[col].isnull().sum())
    print("Min:", df[col].min())
    print("Max:", df[col].max())
    print()

Price columns after cleaning:

min_price:
Data type: float64
Missing values: 7896
Min: 1777.33
Max: 6950.69

max_price:
Data type: float64
Missing values: 7832
Min: 1897.71
Max: 8643.54

modal_price:
Data type: float64
Missing values: 7786
Min: 1845.03
Max: 7716.57

msp:
Data type: float64
Missing values: 8490
Min: 2090.0
Max: 6620.0



In [ ]:
# Review how many price values are still missing after numeric conversion.
# These missing values may need to be imputed or reviewed separately.
print("Missing price values after cleaning:\n")

for col in price_columns:
    print(f"{col}: {df[col].isnull().sum()}")

Missing price values after cleaning:

min_price: 7896
max_price: 7832
modal_price: 7786
msp: 8490


In [ ]:
# Summarize the numeric price columns to understand the distribution and detect outliers.
print("Price statistics:\n")
print(df[price_columns].describe())

Price statistics:

         min_price    max_price  modal_price          msp
count  4104.000000  4168.000000  4214.000000  3510.000000
mean   3574.546607  4072.023720  3795.467587  3726.772080
std    1729.877595  1996.168370  1868.018561  1795.357115
min    1777.330000  1897.710000  1845.030000  2090.000000
25%    2073.177665  2374.197500  2218.425000  2183.000000
50%    3025.359465  2911.240000  2574.601147  3500.000000
75%    5423.305921  6223.625326  5819.330447  5650.000000
max    6950.690000  8643.540000  7716.570000  6620.000000


In [ ]:
# Convert the date column to datetime format so it can be analyzed consistently.
# Invalid date strings are coerced to NaT for later review.
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Invalid dates:", df["date"].isna().sum())
print(df["date"].head(10))

Invalid dates: 10213
0   2026-07-26
1          NaT
2          NaT
3   2026-05-24
4          NaT
5          NaT
6          NaT
7          NaT
8          NaT
9          NaT
Name: date, dtype: datetime64[us]


In [ ]:
# Re-load the raw records to inspect the original date strings before custom parsing.
# This helps us understand the date format variants present in the source data.
import json
import pandas as pd

with open("track3_price_and_msp.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df["date"].head(20).to_list())

['2026/07/26', '2026-07-17', '09.01.2026', '2026/05/24', '18/08/2026', '01-24-2026', '08-Aug-2026', '2026-01-06', '12-Aug-2026', '09.02.2026', '10/02/2026', '24.06.2026', '2026-04-14', '25-Jun-2026', '06/04/2026', '24.01.2026', '06-17-2026', '17/01/2026', '26/03/2026', '2026-04-29']


In [ ]:
# Define a flexible date parser to handle multiple input formats in the dataset.
# This is needed because the raw JSON includes dates in various string patterns.
def clean_date(x):
    if pd.isna(x):
        return pd.NaT
    
    x = str(x).strip()
    
    formats = [
        "%Y/%m/%d",
        "%Y-%m-%d",
        "%d.%m.%Y",
        "%d/%m/%Y",
        "%m-%d-%Y",
        "%d-%m-%Y",
        "%d-%b-%Y",
        "%d-%B-%Y"
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue
    
    return pd.NaT


# Apply the custom parser to normalize every date value to a consistent datetime format.
df["date"] = df["date"].apply(clean_date)

print("Invalid dates after cleaning:", df["date"].isna().sum())
print(df["date"].head(20))


Invalid dates after cleaning: 0
0    2026-07-26
1    2026-07-17
2    2026-01-09
3    2026-05-24
4    2026-08-18
5    2026-01-24
6    2026-08-08
7    2026-01-06
8    2026-08-12
9    2026-02-09
10   2026-02-10
11   2026-06-24
12   2026-04-14
13   2026-06-25
14   2026-04-06
15   2026-01-24
16   2026-06-17
17   2026-01-17
18   2026-03-26
19   2026-04-29
Name: date, dtype: datetime64[us]


In [ ]:
# Check the cleaned date range to confirm the dataset spans a valid temporal period.
print("Date range:")
print("Start:", df["date"].min())
print("End:", df["date"].max())

print("\nMissing dates:", df["date"].isna().sum())

Date range:
Start: 2026-01-01 00:00:00
End: 2026-09-09 00:00:00

Missing dates: 0


In [ ]:
# Review crop categories before standardization to identify naming variants.
# Different spellings such as English, Hindi, and aliases need to be normalized.
print("Unique crop names:", df["crop_name"].nunique())

print("\nCrop names and their counts:")
print(df["crop_name"].value_counts(dropna=False))

Unique crop names: 36

Crop names and their counts:
crop_name
Kapas        442
Ganna        442
गन्ना        415
Sarso        414
सरसों        412
Narma        412
sugarcane    404
Sarson       403
Cotton       394
Sugarcane    394
mustard      394
Mustard      388
cotton       386
Maize        365
कपास         362
Ganne        359
मक्का        356
corn         350
Corn         340
Makka        327
Makki        326
गेहूं        299
WHEAT        294
Wheat        292
wheat        291
GEHUN        273
Dhaan        271
Gehun        268
Kanak        266
Basmati      256
paddy        243
Paddy        242
धान          240
Chawal       233
चावल         225
Rice         222
Name: count, dtype: int64


In [ ]:
# Inspect the raw crop spellings so a mapping can be applied consistently.
# This step is essential to merge variant names into canonical crop categories.
print(df["crop_name"].unique())

<StringArray>
[    'Kapas',      'कपास',     'Dhaan',      'corn',    'Cotton',     'Kanak',
     'WHEAT',     'Gehun',     'गन्ना',     'सरसों',     'Ganna', 'Sugarcane',
   'Basmati',     'Makka',    'cotton', 'sugarcane',   'mustard',     'wheat',
   'Mustard',      'Corn',     'Ganne',     'Maize',       'धान',      'चावल',
     'GEHUN',     'paddy',     'Narma',     'गेहूं',     'Paddy',      'Rice',
     'Wheat',     'मक्का',     'Makki',     'Sarso',    'Sarson',    'Chawal']
Length: 36, dtype: str


In [ ]:
# Map multiple crop aliases to a single standard crop name.
# This reduces duplicates caused by spelling variations across English and regional names.
crop_mapping = {
    "Kapas": "Cotton",
    "कपास": "Cotton",
    "Cotton": "Cotton",
    "cotton": "Cotton",
    "Narma": "Cotton",

    "Ganna": "Sugarcane",
    "Ganne": "Sugarcane",
    "Sugarcane": "Sugarcane",
    "sugarcane": "Sugarcane",
    "गन्ना": "Sugarcane",

    "Sarso": "Mustard",
    "Sarson": "Mustard",
    "mustard": "Mustard",
    "Mustard": "Mustard",
    "सरसों": "Mustard",

    "Kanak": "Wheat",
    "Gehun": "Wheat",
    "GEHUN": "Wheat",
    "Wheat": "Wheat",
    "wheat": "Wheat",
    "WHEAT": "Wheat",
    "गेहूं": "Wheat",

    "Makka": "Maize",
    "Makki": "Maize",
    "Maize": "Maize",
    "Corn": "Maize",
    "corn": "Maize",
    "मक्का": "Maize",

    "Dhaan": "Paddy",
    "धान": "Paddy",
    "paddy": "Paddy",
    "Paddy": "Paddy",

    "Basmati": "Basmati",

    "Chawal": "Rice",
    "Rice": "Rice",
    "चावल": "Rice"
}

df["crop_name"] = df["crop_name"].map(crop_mapping)

print("Unique crop names after cleaning:")
print(df["crop_name"].unique())

print("\nNumber of unique crops:", df["crop_name"].nunique())

Unique crop names after cleaning:
<StringArray>
['Cotton', 'Paddy', 'Maize', 'Wheat', 'Sugarcane', 'Mustard', 'Basmati',
 'Rice']
Length: 8, dtype: str

Number of unique crops: 8


In [ ]:
# Confirm whether any crop names remain missing after mapping.
# Rows with missing crop labels may need further review or a fallback category.
print("Missing crop names:", df["crop_name"].isna().sum())
print("\nCrop counts:")
print(df["crop_name"].value_counts())

Missing crop names: 0

Crop counts:
crop_name
Maize        2064
Sugarcane    2014
Mustard      2011
Cotton       1996
Wheat        1983
Paddy         996
Rice          680
Basmati       256
Name: count, dtype: int64


In [ ]:
# Load the cleaned mandi master as a reference dataset for cross-validation.
# This is used to compare mandi IDs and district names with the official mandi table.
mandi_master = pd.read_csv("cleaned_mandi_master.csv")

print("Mandi Master shape:", mandi_master.shape)
print(mandi_master.head())

Mandi Master shape: (57, 6)
   Mandi_ID             Mandi_Name  District    State Mandi_Type  \
0  MANDI001        Hyderabad Mandi  Ludhiana   Punjab    PRIVATE   
1  MANDI002          Solapur Mandi  Ludhiana  Unknown       APMC   
2  MANDI003       Vijayawada Mandi  Ludhiana   Punjab       APMC   
3  MANDI004   Khandwa Grain Market  Ludhiana   Punjab       APMC   
4  MANDI005  Bhilwara Grain Market  Amritsar   Punjab     DIRECT   

   Total_Area_Acres  
0              11.0  
1              37.0  
2              19.0  
3              15.0  
4              29.0  


In [ ]:
# Join the cleaned price dataset to the mandi master using mandi_id first and district as fallback.
df["mandi_id"] = df["mandi_id"].astype("string").str.strip().str.upper()
df["district"] = df["district"].astype("string").str.strip().str.title()

master = mandi_master.copy()
master["Mandi_ID"] = master["Mandi_ID"].astype("string").str.strip().str.upper()
master["District"] = master["District"].astype("string").str.strip().str.title()

master_by_id = master[["Mandi_ID", "Mandi_Name", "State", "Mandi_Type", "Total_Area_Acres"]].rename(
    columns={"Mandi_ID": "mandi_id"}
)

master_by_district = master[["District", "Mandi_Name", "State", "Mandi_Type", "Total_Area_Acres"]].drop_duplicates().rename(
    columns={
        "District": "district",
        "Mandi_Name": "Mandi_Name_district",
        "State": "State_district",
        "Mandi_Type": "Mandi_Type_district",
        "Total_Area_Acres": "Total_Area_Acres_district",
    }
)

df = df.merge(master_by_id, on="mandi_id", how="left")
df = df.merge(master_by_district, on="district", how="left")

# Prefer the exact mandi_id match; if none is found, use the district match.
for primary, fallback in [
    ("Mandi_Name", "Mandi_Name_district"),
    ("State", "State_district"),
    ("Mandi_Type", "Mandi_Type_district"),
    ("Total_Area_Acres", "Total_Area_Acres_district"),
]:
    df[primary] = df[primary].combine_first(df[fallback])

# Keep the final master metadata columns explicit for analysis and export.
df["Mandi_ID"] = df["mandi_id"]
df["District"] = df["district"]

print("Join summary:")
print("Matched by mandi_id:", df["Mandi_Name"].notna().sum())
print("Still missing master fields after join:", df["Mandi_Name"].isna().sum())

# Save the joined dataset for downstream use.
df.to_csv("price_and_msp_joined_mandi_master.csv", index=False)
print("Joined dataset saved to price_and_msp_joined_mandi_master.csv")


In [ ]:
# Standardize district names to keep text values consistent and comparable.
# This trims whitespace and normalizes to title case for downstream joins.
df["district"] = df["district"].astype("string").str.strip().str.title()

print("Missing districts:", df["district"].isna().sum())
print("\nDistrict values:")
print(df["district"].value_counts(dropna=False))

Missing districts: 773

District values:
district
Kurukshetra    843
Ludhiana       841
Ferozepur      832
Ambala         831
Moga           809
Jalandhar      807
Fatehabad      803
Bathinda       799
Amritsar       791
Hisar          783
Patiala        780
Sirsa          774
<NA>           773
               773
Karnal         761
Name: count, dtype: Int64


In [ ]:
# Fill any remaining district missing values with a default placeholder.
# This prevents missing geographic labels from breaking filters or group-by operations.
df["district"] = df["district"].fillna("Unknown")

print("Missing districts:", df["district"].isna().sum())
print("\nDistrict values:")
print(df["district"].value_counts())

Missing districts: 0

District values:
district
Kurukshetra    843
Ludhiana       841
Ferozepur      832
Ambala         831
Moga           809
Jalandhar      807
Fatehabad      803
Bathinda       799
Amritsar       791
Hisar          783
Patiala        780
Sirsa          774
Unknown        773
               773
Karnal         761
Name: count, dtype: Int64


In [ ]:
# Replace blank district strings with a default value to handle whitespace-only entries.
# This catches edge cases where fields are present but effectively empty.
df["district"] = df["district"].replace(r"^\s*$", "Unknown", regex=True)

print("Missing districts:", df["district"].isna().sum())
print("\nDistrict values:")
print(df["district"].value_counts())

Missing districts: 0

District values:
district
Unknown        1546
Kurukshetra     843
Ludhiana        841
Ferozepur       832
Ambala          831
Moga            809
Jalandhar       807
Fatehabad       803
Bathinda        799
Amritsar        791
Hisar           783
Patiala         780
Sirsa           774
Karnal          761
Name: count, dtype: Int64


In [ ]:
# Check whether duplicate rows still remain after the cleaning steps.
# Duplicate records can distort agricultural summaries and pricing trends.
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [ ]:
# Final validation of the cleaned dataset before saving or analysis.
# This confirms the schema, null counts, and data types are acceptable.
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape: (12000, 9)

Missing values:
record_id         0
date              0
mandi_id       1235
district          0
crop_name         0
min_price         0
max_price         0
modal_price       0
msp               0
dtype: int64

Data types:
record_id                 str
date           datetime64[us]
mandi_id                  str
district               string
crop_name                 str
min_price              object
max_price              object
modal_price            object
msp                    object
dtype: object


In [ ]:
# Apply the same price-cleaning function again to ensure the numeric columns remain consistent.
# This is useful after other transformations to keep the price fields standardized.
price_columns = ["min_price", "max_price", "modal_price", "msp"]

for col in price_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace("INR", "", regex=False)
        .str.replace("/-", "", regex=False)
        .str.strip()
    )
    
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[price_columns].dtypes)

min_price      float64
max_price      float64
modal_price    float64
msp            float64
dtype: object


In [ ]:
# Preview the cleaned dataset after the main transformations.
# This gives a final visual check before export or deeper analysis.
print(df.head(10))

  record_id       date   mandi_id     district  crop_name    min_price  \
0  PR001650 2026-07-26   MANDI005      Unknown     Cotton  6850.985818   
1  PR000246 2026-07-17   MANDI028    Fatehabad     Cotton          NaN   
2  PR010091 2026-01-09   MANDI011    Jalandhar      Paddy          NaN   
3  PR000982 2026-05-24       M012    Ferozepur      Maize          NaN   
4  PR001708 2026-08-18        013       Karnal     Cotton          NaN   
5  PR004312 2026-01-24   MANDI001      Patiala     Cotton          NaN   
6  PR000288 2026-08-08       M031        Hisar      Wheat          NaN   
7  PR002536 2026-01-06       M036  Kurukshetra      Wheat          NaN   
8  PR009427 2026-08-12  MANDI-050      Unknown      Wheat  2277.530000   
9  PR003418 2026-02-09       M011       Ambala  Sugarcane          NaN   

     max_price  modal_price     msp  
0          NaN      7210.58     NaN  
1  7654.180000          NaN     NaN  
2  2013.180000          NaN     NaN  
3          NaN          NaN     N

In [ ]:
# Expand the display to inspect all columns in the cleaned result.
# This helps confirm that the final table structure is ready for analysis.
pd.set_option("display.max_columns", None)
print(df.head(10))

  record_id       date   mandi_id     district  crop_name    min_price  \
0  PR001650 2026-07-26   MANDI005      Unknown     Cotton  6850.985818   
1  PR000246 2026-07-17   MANDI028    Fatehabad     Cotton          NaN   
2  PR010091 2026-01-09   MANDI011    Jalandhar      Paddy          NaN   
3  PR000982 2026-05-24       M012    Ferozepur      Maize          NaN   
4  PR001708 2026-08-18        013       Karnal     Cotton          NaN   
5  PR004312 2026-01-24   MANDI001      Patiala     Cotton          NaN   
6  PR000288 2026-08-08       M031        Hisar      Wheat          NaN   
7  PR002536 2026-01-06       M036  Kurukshetra      Wheat          NaN   
8  PR009427 2026-08-12  MANDI-050      Unknown      Wheat  2277.530000   
9  PR003418 2026-02-09       M011       Ambala  Sugarcane          NaN   

     max_price  modal_price     msp  
0          NaN      7210.58     NaN  
1  7654.180000          NaN     NaN  
2  2013.180000          NaN     NaN  
3          NaN          NaN     N